# Crop Disease Detection using Lighter MobileNetV2

This notebook trains a lighter Convolutional Neural Network (CNN) based on MobileNetV2 using a subset of the PlantVillage dataset to avoid RAM overflow on Colab.

In [ ]:
!pip install -q tensorflow-datasets

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
import numpy as np

print("TensorFlow version:", tf.__version__)

## 1. Load the PlantVillage Dataset & Filter Subset

To save RAM, we will only use the first 5 classes of the dataset.

In [ ]:
# Load the dataset using TFDS
dataset, info = tfds.load('plant_village', with_info=True, as_supervised=True)

# Get all class names
all_class_names = info.features['label'].names

# Filter dataset to only keep the first 5 classes to save memory
NUM_CLASSES = 5
class_names = all_class_names[:NUM_CLASSES]

def filter_classes(image, label):
    return label < NUM_CLASSES

# Apply filter
filtered_ds = dataset['train'].filter(filter_classes)

# Count elements after filtering (this is a slow operation, but needed to know sizes)
DATASET_SIZE = filtered_ds.reduce(0, lambda x, _: x + 1).numpy()
print(f"Total images in subset: {DATASET_SIZE}")

train_size = int(0.8 * DATASET_SIZE)
val_size = int(0.2 * DATASET_SIZE)

full_ds = filtered_ds.shuffle(10000, seed=42)
train_ds = full_ds.take(train_size)
val_ds = full_ds.skip(train_size)

print(f"Subset classes: {class_names}")
print(f"Training samples: {train_size}")
print(f"Validation samples: {val_size}")

## 2. Preprocess Data with Disk Caching

Using `.cache('filename')` forces tf.data to cache the preprocessed images to the Colab disk instead of RAM, avoiding out-of-memory crashes.

In [ ]:
IMG_SIZE = 224 # MobileNetV2 expects 224x224
BATCH_SIZE = 16 # Reduced batch size to save memory

def format_image(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = image / 255.0 # Normalize to [0,1]
    return image, label

AUTOTUNE = tf.data.AUTOTUNE

train_batches = (
    train_ds
    .map(format_image, num_parallel_calls=AUTOTUNE)
    .cache('train_cache')  # Cache to DISK instead of RAM
    .shuffle(1000)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

validation_batches = (
    val_ds
    .map(format_image, num_parallel_calls=AUTOTUNE)
    .cache('val_cache')   # Cache to DISK instead of RAM
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

## 3. Build the Lighter Model (MobileNetV2 with alpha=0.35)

Setting `alpha=0.35` makes the network channels significantly narrower, massively reducing model size and RAM usage during training.

In [ ]:
IMG_SHAPE = (IMG_SIZE, IMG_SIZE, 3)

# Create the base model from the pre-trained model MobileNet V2
base_model = tf.keras.applications.MobileNetV2(input_shape=IMG_SHAPE,
                                               include_top=False,
                                               weights='imagenet',
                                               alpha=0.35) # Using lighter alpha multiplier

# Freeze the base_model
base_model.trainable = False

# Add classification head
global_average_layer = tf.keras.layers.GlobalAveragePooling2D()
prediction_layer = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')

model = tf.keras.Sequential([
  base_model,
  global_average_layer,
  tf.keras.layers.Dropout(0.2),
  prediction_layer
])

model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

model.summary()

## 4. Train the Model

In [ ]:
EPOCHS = 10

# Use EarlyStopping and ModelCheckpoint to save the best model
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint('plant_disease_model.h5', save_best_only=True)
]

history = model.fit(train_batches,
                    epochs=EPOCHS,
                    validation_data=validation_batches,
                    callbacks=callbacks)

## 5. Evaluate and Save

In [ ]:
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

plt.figure(figsize=(8, 8))
plt.subplot(2, 1, 1)
plt.plot(acc, label='Training Accuracy')
plt.plot(val_acc, label='Validation Accuracy')
plt.legend(loc='lower right')
plt.ylabel('Accuracy')
plt.ylim([min(plt.ylim()),1])
plt.title('Training and Validation Accuracy')

plt.subplot(2, 1, 2)
plt.plot(loss, label='Training Loss')
plt.plot(val_loss, label='Validation Loss')
plt.legend(loc='upper right')
plt.ylabel('Cross Entropy')
plt.title('Training and Validation Loss')
plt.xlabel('epoch')
plt.show()

In [ ]:
# Save class names for inference later
import json
with open('class_indices.json', 'w') as f:
    json.dump(class_names, f)

print("Model saved as plant_disease_model.h5")
print("Class indices saved as class_indices.json")

## 6. Download Files to Local Machine
Run this cell to download the trained model and class indices if you are on Google Colab.

In [ ]:
try:
  from google.colab import files
  files.download('plant_disease_model.h5')
  files.download('class_indices.json')
except:
  print("Not running in Colab, skip downloading.")